# Proteomics Summary Literature Evidence Agent

This notebook is adapted for per-drug proteomics summary Markdown files, such as `6-Mercaptopurine.summary.md`. It parses each drug summary, extracts expected findings, unexpected findings, and follow-up hypotheses, then searches PubMed for literature evidence.

Key behavior:

1. Search each claim/hypothesis using drug + mechanism/pathway/protein keywords.
2. Use fallback queries when direct evidence is sparse.
3. Score papers as Strong, Moderate, Weak, Background, or No direct evidence found.
4. Explicitly preserve claims with no evidence instead of forcing unsupported matches.
5. Export CSV tables and Markdown reports for manual review.

## 1. Imports

In [1]:
# !pip install pandas requests tqdm openpyxl nbformat

import re
import time
import json
import textwrap
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from xml.etree import ElementTree as ET

import pandas as pd
import requests
from tqdm.auto import tqdm

## 2. User settings

Change `SUMMARY_DIR` to the folder containing your `*.summary.md` files. For a first test, you can set it to the folder containing the example file.

In [12]:
# =========================
# Input / output paths
# =========================

SUMMARY_DIR = Path("D:/project8  Robotics and AI enable automation in modern proteomics/drug_id_cards_MAS_results/test")  # folder containing drug summary Markdown files
SUMMARY_GLOB = "*.summary*.md"  # matches e.g. 6-Mercaptopurine.summary.md or .summary(1).md
OUTPUT_DIR = Path("D:/project8  Robotics and AI enable automation in modern proteomics/drug_id_cards_MAS_results/test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# PubMed / NCBI settings
# =========================

NCBI_EMAIL = "your.email@domain.com"  # replace with your email; recommended by NCBI
NCBI_TOOL = "proteomics_summary_literature_agent"
REQUEST_SLEEP_SECONDS = 0.34  # polite rate limit without NCBI API key
MAX_PAPERS_PER_CLAIM = 10
MIN_PUBLICATION_YEAR = 1990  # use None to remove date filter

# Optional context terms. Keep this broad; too many constraints can hide useful evidence.
CONTEXT_KEYWORDS = ["HepG2", "hepatocyte", "liver", "cancer", "cell line"]

# Search terms that bias toward molecular mechanisms, not just clinical use.
MECHANISM_KEYWORDS = [
    "proteomics", "protein expression", "pathway", "signaling", "mechanism",
    "cell cycle", "apoptosis", "stress response", "metabolism"
]

# Output files
CLAIMS_TABLE_FILE = OUTPUT_DIR / "parsed_summary_claims.csv"
EVIDENCE_TABLE_FILE = OUTPUT_DIR / "literature_evidence_long.csv"
CLAIM_SUMMARY_FILE = OUTPUT_DIR / "literature_evidence_by_claim.csv"
MARKDOWN_REPORT_FILE = OUTPUT_DIR / "literature_evidence_report.md"
QUERY_LOG_FILE = OUTPUT_DIR / "query_log.json"

## 3. Text parsing utilities

In [13]:
def clean_text(x) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def normalize_dash(text: str) -> str:
    return str(text).replace("‑", "-").replace("–", "-").replace("—", "-").replace("−", "-")


def infer_drug_from_markdown(text: str, file_path: Path) -> str:
    m = re.search(r"^#\s*Drug summary:\s*(.+?)\s*$", text, flags=re.I | re.M)
    if m:
        return clean_text(m.group(1))
    name = file_path.name
    name = re.sub(r"\.summary.*\.md$", "", name, flags=re.I)
    return clean_text(name)


def split_markdown_sections(text: str) -> Dict[str, str]:
    """Return top-level section title -> section body for ## sections."""
    text = normalize_dash(text)
    matches = list(re.finditer(r"^##\s+(.+?)\s*$", text, flags=re.M))
    sections = {}
    for i, m in enumerate(matches):
        title = clean_text(m.group(1))
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[title] = text[start:end].strip()
    return sections


def get_bullets(section_text: str) -> List[str]:
    bullets = []
    for line in section_text.splitlines():
        line = line.strip()
        if re.match(r"^[-*]\s+", line):
            bullets.append(re.sub(r"^[-*]\s+", "", line).strip())
    return bullets


def parse_hypotheses(section_text: str) -> List[Dict[str, str]]:
    """Parse ### Hypothesis blocks and return one row per hypothesis."""
    rows = []
    parts = re.split(r"^###\s+Hypothesis\s+(\d+)\s*$", section_text, flags=re.I | re.M)
    # parts: preamble, number, body, number, body...
    for i in range(1, len(parts), 2):
        number = parts[i]
        body = parts[i + 1]
        row = {"hypothesis_number": number, "hypothesis": "", "why_interesting": "", "suggested_validation": "", "priority": ""}
        for line in body.splitlines():
            line = line.strip()
            if not line.startswith("-"):
                continue
            line = re.sub(r"^-\s*", "", line)
            m = re.match(r"\*\*(.+?)\*\*:\s*(.*)", line)
            if not m:
                continue
            key = m.group(1).strip().lower().replace(" ", "_")
            value = clean_text(m.group(2))
            if key in row:
                row[key] = value
        if row["hypothesis"]:
            rows.append(row)
    return rows


def extract_gene_symbols(text: str) -> List[str]:
    """Extract likely human gene/protein symbols from a claim. Conservative but useful for query building."""
    text = normalize_dash(text)
    # Common pathway words to exclude even if uppercase.
    stop = {
        "DNA", "RNA", "GO", "EMT", "ECM", "UPR", "ROS", "ER", "FDR", "LC", "MS", "ELISA",
        "SASP", "HIF", "NF", "PI3K", "AKT", "IGF", "ATR", "CHK", "KEGG"
    }
    symbols = re.findall(r"\b[A-Z][A-Z0-9]{1,9}\b", text)
    symbols = [s for s in symbols if s not in stop and not re.fullmatch(r"\d+", s)]
    # handle slash groups like IGFBP1/3, E2F6/8, SLC22A3/SLC13A5 already partly captured
    slash = re.findall(r"\b([A-Z]{2,}\d+)/(\d+)\b", text)
    for prefix_num, suffix in slash:
        prefix = re.sub(r"\d+$", "", prefix_num)
        symbols.append(prefix_num)
        symbols.append(prefix + suffix)
    return list(dict.fromkeys(symbols))[:12]


def extract_mechanism_terms(text: str) -> List[str]:
    """Extract mechanism/pathway phrases likely to be useful in PubMed queries."""
    t = normalize_dash(text)
    candidates = []
    keyword_patterns = [
        r"cell-cycle", r"cell cycle", r"DNA replication", r"mitotic", r"mitosis", r"E2F", r"G2M",
        r"NF.?kB", r"TNFA", r"inflammatory", r"apoptosis", r"UPR", r"unfolded protein response",
        r"hypoxia", r"EMT", r"ECM remodeling", r"angiogenesis", r"IGF", r"PI3K.?AKT",
        r"amino acid import", r"lipid", r"cholesterol", r"sterol", r"solute", r"transporter",
        r"autophagy", r"senescence", r"secretory", r"stress response", r"ER stress"
    ]
    for pat in keyword_patterns:
        if re.search(pat, t, flags=re.I):
            candidates.append(re.sub(r"\\.?", "", pat))
    # Pull quoted or parenthetical hallmark/GO-like names, then normalize underscores.
    for m in re.findall(r"\b(?:Hallmark|GO)\s+([A-Za-z0-9_/-]+)", t, flags=re.I):
        candidates.append(m.replace("_", " "))
    return list(dict.fromkeys([clean_text(c) for c in candidates if clean_text(c)]))[:12]


def claim_text_from_row(row: Dict[str, str]) -> str:
    return clean_text(" ".join([row.get("claim", ""), row.get("hypothesis", ""), row.get("why_interesting", "")]))

## 4. Parse drug summary Markdown files

In [14]:
def parse_summary_file(file_path: Path) -> List[Dict[str, str]]:
    text = file_path.read_text(encoding="utf-8", errors="replace")
    drug = infer_drug_from_markdown(text, file_path)
    sections = split_markdown_sections(text)
    rows = []

    # Executive summary becomes one broad claim, useful for drug-level search.
    executive = clean_text(sections.get("Executive summary", ""))
    if executive:
        rows.append({
            "source_file": str(file_path),
            "drug": drug,
            "claim_type": "executive_summary",
            "claim_id": f"{drug}__executive_summary",
            "claim": executive,
            "hypothesis": "",
            "why_interesting": "",
            "suggested_validation": "",
            "priority": "",
        })

    # Bullet findings.
    for section_name in ["Findings consistent with expected biology", "Unexpected but plausible findings"]:
        for j, bullet in enumerate(get_bullets(sections.get(section_name, "")), start=1):
            rows.append({
                "source_file": str(file_path),
                "drug": drug,
                "claim_type": section_name.lower().replace(" ", "_"),
                "claim_id": f"{drug}__{section_name}__{j}",
                "claim": bullet,
                "hypothesis": "",
                "why_interesting": "",
                "suggested_validation": "",
                "priority": "",
            })

    # Hypotheses.
    for h in parse_hypotheses(sections.get("Follow-up hypotheses", "")):
        rows.append({
            "source_file": str(file_path),
            "drug": drug,
            "claim_type": "follow_up_hypothesis",
            "claim_id": f"{drug}__hypothesis_{h.get('hypothesis_number')}",
            "claim": h.get("hypothesis", ""),
            "hypothesis": h.get("hypothesis", ""),
            "why_interesting": h.get("why_interesting", ""),
            "suggested_validation": h.get("suggested_validation", ""),
            "priority": h.get("priority", ""),
        })

    # Add parsed keyword columns.
    for r in rows:
        combined = claim_text_from_row(r)
        r["genes"] = ";".join(extract_gene_symbols(combined))
        r["mechanism_terms"] = ";".join(extract_mechanism_terms(combined))
    return rows


summary_files = sorted(SUMMARY_DIR.glob(SUMMARY_GLOB))
print(f"Found {len(summary_files)} summary files")

all_claims = []
for fp in summary_files:
    all_claims.extend(parse_summary_file(fp))

claims_df = pd.DataFrame(all_claims)
claims_df.to_csv(CLAIMS_TABLE_FILE, index=False)
print(claims_df.shape)
claims_df.head(20)

Found 1 summary files
(7, 11)


,source_file,drug,claim_type,claim_id,claim,hypothesis,why_interesting,suggested_validation,priority,genes,mechanism_terms
0,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,executive_summary,6-Mercaptopurine__executive_summary,Proteomic profiling under 6-mercaptopurine sho...,,,,,CDC7;EXO1;E2F6;HYPOXIA;GDF15;IGFBP1;IL4R;TNFRS...,DNA replication;mitotic;mitosis;E2F;G2M;NF.?kB...
1,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,findings_consistent_with_expected_biology,6-Mercaptopurine__Findings consistent with exp...,Strong suppression of DNA replication and mito...,,,,,CDC7;EXO1;E2F6;E2F8;STIL,cell-cycle;DNA replication;mitotic;mitosis;E2F...
2,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,findings_consistent_with_expected_biology,6-Mercaptopurine__Findings consistent with exp...,Induction of inflammatory/NFκB and cellular st...,,,,,APOPTOSIS;GDF15;IGFBP1;IL4R;TNFRSF12A;JUNB;AGR...,NF.?kB;TNFA;inflammatory;apoptosis;UPR;IGF;INF...
3,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,findings_consistent_with_expected_biology,6-Mercaptopurine__Findings consistent with exp...,Activation of hypoxia-like responses (Hallmark...,,,,,HYPOXIA;LOXL2;NRP2;TYRO3;GDF15,hypoxia;HYPOXIA
4,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,unexpected_but_plausible_findings,6-Mercaptopurine__Unexpected but plausible fin...,Pronounced EMT/ECM remodeling and angiogenesis...,,,,,LOXL2;PLAUR;CEMIP;LAMC3;CTHRC1;NRP2,EMT;ECM remodeling;angiogenesis;EMT/tissue
5,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,unexpected_but_plausible_findings,6-Mercaptopurine__Unexpected but plausible fin...,Evidence for adaptive signaling and nutrient u...,,,,,IGFBP1;IGFBP3,IGF;PI3K.?AKT;amino acid import
6,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,unexpected_but_plausible_findings,6-Mercaptopurine__Unexpected but plausible fin...,Potential metabolic/transport reprogramming: H...,,,,,HMGCR;SLC22A3;SLC13A5;SLC35C1,sterol;solute;transporter


## 5. PubMed query construction

The query strategy is intentionally tiered. The agent tries direct drug-mechanism/protein evidence first, then broader mechanism/context evidence. A claim is allowed to remain unsupported when no convincing papers are found.

In [15]:
def quote_pubmed_phrase(term: str) -> str:
    term = clean_text(term)
    if not term:
        return ""
    # Do not quote gene symbols; quote multi-word phrases.
    if len(term.split()) > 1:
        return f'"{term}"'
    return term


def make_or_block(terms: List[str], max_terms: int = 8) -> str:
    terms = [quote_pubmed_phrase(t) for t in terms if clean_text(t)]
    terms = list(dict.fromkeys(terms))[:max_terms]
    if not terms:
        return ""
    if len(terms) == 1:
        return terms[0]
    return "(" + " OR ".join(terms) + ")"


def drug_aliases(drug: str) -> List[str]:
    aliases = [drug]
    d = drug.lower().replace(" ", "")
    if d in {"6-mercaptopurine", "6mercaptopurine", "mercaptopurine"}:
        aliases += ["mercaptopurine", "6-MP", "6MP"]
    return list(dict.fromkeys([a for a in aliases if a]))


def build_pubmed_queries_for_claim(row: pd.Series) -> Dict[str, str]:
    drug = clean_text(row.get("drug", ""))
    claim = clean_text(row.get("claim", ""))
    genes = [x for x in clean_text(row.get("genes", "")).split(";") if x]
    mechanisms = [x for x in clean_text(row.get("mechanism_terms", "")).split(";") if x]

    drug_block = make_or_block(drug_aliases(drug), max_terms=6)
    gene_block = make_or_block(genes, max_terms=8)
    mech_block = make_or_block(mechanisms, max_terms=8)
    context_block = make_or_block(CONTEXT_KEYWORDS, max_terms=8)
    mechanism_bias = make_or_block(MECHANISM_KEYWORDS, max_terms=8)

    queries = {}

    # Highest value: drug + mechanism/protein.
    if drug_block and mech_block:
        queries["drug_mechanism"] = f"{drug_block} AND {mech_block}"
    if drug_block and gene_block:
        queries["drug_gene_or_protein"] = f"{drug_block} AND {gene_block}"
    if drug_block and mech_block and context_block:
        queries["drug_mechanism_context"] = f"{drug_block} AND {mech_block} AND {context_block}"
    if drug_block and gene_block and mechanism_bias:
        queries["drug_gene_mechanistic"] = f"{drug_block} AND {gene_block} AND {mechanism_bias}"

    # Broad drug biology; useful when the claim is global or many proteins are listed.
    if drug_block and mechanism_bias:
        queries["drug_mechanistic_biology"] = f"{drug_block} AND {mechanism_bias}"

    # Background evidence when direct drug evidence does not exist.
    if mech_block and context_block:
        queries["mechanism_context_background"] = f"{mech_block} AND {context_block}"
    if gene_block and context_block:
        queries["gene_context_background"] = f"{gene_block} AND {context_block}"
    if mech_block:
        queries["mechanism_background"] = mech_block

    # Add publication year filter.
    if MIN_PUBLICATION_YEAR:
        for k in list(queries):
            queries[k] = f'({queries[k]}) AND ("{MIN_PUBLICATION_YEAR}"[Date - Publication] : "3000"[Date - Publication])'
    return queries


if len(claims_df):
    test_row = claims_df.iloc[0]
    print(test_row[["drug", "claim_type", "genes", "mechanism_terms"]])
    print("\nQueries:")
    for k, q in build_pubmed_queries_for_claim(test_row).items():
        print(f"[{k}] {q}")

drug                                                6-Mercaptopurine
claim_type                                         executive_summary
genes              CDC7;EXO1;E2F6;HYPOXIA;GDF15;IGFBP1;IL4R;TNFRS...
mechanism_terms    DNA replication;mitotic;mitosis;E2F;G2M;NF.?kB...
Name: 0, dtype: object

Queries:
[drug_mechanism] ((6-Mercaptopurine OR mercaptopurine OR 6-MP OR 6MP) AND ("DNA replication" OR mitotic OR mitosis OR E2F OR G2M OR NF.?kB OR TNFA OR inflammatory)) AND ("1990"[Date - Publication] : "3000"[Date - Publication])
[drug_gene_or_protein] ((6-Mercaptopurine OR mercaptopurine OR 6-MP OR 6MP) AND (CDC7 OR EXO1 OR E2F6 OR HYPOXIA OR GDF15 OR IGFBP1 OR IL4R OR TNFRSF12A)) AND ("1990"[Date - Publication] : "3000"[Date - Publication])
[drug_mechanism_context] ((6-Mercaptopurine OR mercaptopurine OR 6-MP OR 6MP) AND ("DNA replication" OR mitotic OR mitosis OR E2F OR G2M OR NF.?kB OR TNFA OR inflammatory) AND (HepG2 OR hepatocyte OR liver OR cancer OR "cell line")) AND ("1990"[Da

## 6. PubMed API functions

In [16]:
def ncbi_esearch(query: str, max_results: int = 10) -> List[str]:
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": max_results,
        "sort": "relevance",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json().get("esearchresult", {}).get("idlist", [])


def ncbi_efetch(pmids: List[str]) -> List[Dict]:
    if not pmids:
        return []
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    records = []
    for article in root.findall(".//PubmedArticle"):
        pmid = clean_text(article.findtext(".//PMID"))
        title = clean_text("".join(article.find(".//ArticleTitle").itertext()) if article.find(".//ArticleTitle") is not None else "")
        journal = clean_text(article.findtext(".//Journal/Title"))
        abstract_parts = []
        for node in article.findall(".//Abstract/AbstractText"):
            label = node.attrib.get("Label", "")
            txt = clean_text("".join(node.itertext()))
            abstract_parts.append(f"{label}: {txt}" if label else txt)
        abstract = clean_text(" ".join(abstract_parts))
        year = clean_text(article.findtext(".//JournalIssue/PubDate/Year") or article.findtext(".//ArticleDate/Year") or "")
        authors = []
        for a in article.findall(".//AuthorList/Author")[:6]:
            last = clean_text(a.findtext("LastName"))
            initials = clean_text(a.findtext("Initials"))
            if last or initials:
                authors.append(clean_text(f"{last} {initials}"))
        pub_types = [clean_text(x.text) for x in article.findall(".//PublicationType") if x.text]
        records.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "journal": journal,
            "year": year,
            "authors": "; ".join(authors),
            "publication_types": "; ".join(pub_types),
            "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else "",
        })
    return records

## 7. Evidence scoring

Strong means the paper matches the drug and a claim-specific mechanism or protein. Background means it may explain the biology but does not directly validate this drug-specific observation.

In [17]:
def contains_term(text: str, term: str) -> bool:
    text = clean_text(text).lower()
    term = clean_text(term).lower()
    if not text or not term:
        return False
    # For short terms/gene symbols, use boundaries.
    if len(term) <= 5 and re.fullmatch(r"[a-z0-9-]+", term):
        return re.search(rf"(?<![a-z0-9]){re.escape(term)}(?![a-z0-9])", text) is not None
    return term in text


def score_evidence(row: pd.Series, paper: Dict, query_type: str) -> Dict:
    drug = clean_text(row.get("drug", ""))
    genes = [x for x in clean_text(row.get("genes", "")).split(";") if x]
    mechanisms = [x for x in clean_text(row.get("mechanism_terms", "")).split(";") if x]
    title_abs = clean_text(paper.get("title", "") + " " + paper.get("abstract", ""))

    score = 0
    reasons = []

    query_bonus = {
        "drug_mechanism": 30,
        "drug_gene_or_protein": 28,
        "drug_mechanism_context": 32,
        "drug_gene_mechanistic": 30,
        "drug_mechanistic_biology": 18,
        "mechanism_context_background": 12,
        "gene_context_background": 10,
        "mechanism_background": 8,
    }.get(query_type, 5)
    score += query_bonus
    reasons.append(f"query={query_type}")

    drug_matched = any(contains_term(title_abs, a) for a in drug_aliases(drug))
    gene_matches = [g for g in genes if contains_term(title_abs, g)]
    mechanism_matches = [m for m in mechanisms if contains_term(title_abs, m)]
    context_matches = [c for c in CONTEXT_KEYWORDS if contains_term(title_abs, c)]

    if drug_matched:
        score += 25
        reasons.append("drug_match")
    if gene_matches:
        score += min(24, 8 * len(gene_matches))
        reasons.append("gene_match:" + ",".join(gene_matches[:4]))
    if mechanism_matches:
        score += min(30, 10 * len(mechanism_matches))
        reasons.append("mechanism_match:" + ",".join(mechanism_matches[:4]))
    if context_matches:
        score += min(10, 4 * len(context_matches))
        reasons.append("context_match:" + ",".join(context_matches[:3]))

    lower = title_abs.lower()
    if any(w in lower for w in ["proteomic", "proteomics", "protein expression", "western blot", "immunoblot", "signaling", "mechanism", "pathway"]):
        score += 8
        reasons.append("mechanistic_assay_terms")
    if not paper.get("abstract"):
        score -= 8
        reasons.append("no_abstract_penalty")

    pub_types = paper.get("publication_types", "").lower()
    if "review" in pub_types:
        score -= 3
        reasons.append("review_less_direct")
    if "clinical trial" in pub_types:
        score += 4
        reasons.append("clinical_trial")

    # Classification emphasizes directness, not just score.
    score = max(0, min(100, score))
    direct = drug_matched and (bool(gene_matches) or bool(mechanism_matches))
    background = (bool(gene_matches) or bool(mechanism_matches)) and not drug_matched

    if direct and score >= 70:
        level = "Strong"
    elif direct and score >= 55:
        level = "Moderate"
    elif direct:
        level = "Weak"
    elif background and score >= 40:
        level = "Background"
    else:
        level = "Low relevance"

    return {
        "evidence_score": score,
        "evidence_level": level,
        "direct_drug_evidence": direct,
        "background_biology_only": background and not direct,
        "score_reasons": "; ".join(reasons),
    }

## 8. Run literature search

In [18]:
def search_literature_for_claim(row: pd.Series, max_papers: int = MAX_PAPERS_PER_CLAIM) -> Tuple[List[Dict], Dict[str, str]]:
    queries = build_pubmed_queries_for_claim(row)
    records_out = []
    seen_pmids = set()

    for query_type, query in queries.items():
        if len(records_out) >= max_papers:
            break
        try:
            pmids = ncbi_esearch(query, max_results=max_papers)
            time.sleep(REQUEST_SLEEP_SECONDS)
            pmids = [p for p in pmids if p not in seen_pmids]
            if not pmids:
                continue
            records = ncbi_efetch(pmids)
            time.sleep(REQUEST_SLEEP_SECONDS)
            for rec in records:
                pmid = rec.get("pmid", "")
                if not pmid or pmid in seen_pmids:
                    continue
                seen_pmids.add(pmid)
                rec = rec.copy()
                rec.update(score_evidence(row, rec, query_type))
                rec["query_type"] = query_type
                rec["query"] = query
                records_out.append(rec)
                if len(records_out) >= max_papers:
                    break
        except Exception as e:
            print(f"Query failed for {row.get('claim_id')}: {query_type} | {e}")

    records_out = sorted(records_out, key=lambda x: x.get("evidence_score", 0), reverse=True)
    return records_out, queries


def run_literature_agent(claims_df: pd.DataFrame, max_claims: Optional[int] = None) -> pd.DataFrame:
    rows = []
    query_log = {}
    work = claims_df.head(max_claims).copy() if max_claims else claims_df.copy()

    for _, claim_row in tqdm(work.iterrows(), total=len(work)):
        records, queries = search_literature_for_claim(claim_row)
        query_log[claim_row["claim_id"]] = queries
        meta = claim_row.to_dict()

        # Keep unsupported claims in the output.
        if not records:
            rows.append({
                **meta,
                "pmid": "",
                "title": "",
                "abstract": "",
                "journal": "",
                "year": "",
                "authors": "",
                "publication_types": "",
                "pubmed_url": "",
                "query_type": "",
                "query": "",
                "evidence_score": 0,
                "evidence_level": "No direct evidence found",
                "direct_drug_evidence": False,
                "background_biology_only": False,
                "score_reasons": "No PubMed result from tiered queries",
            })
            continue

        # Drop clearly low relevance papers unless this would leave no row.
        useful = [r for r in records if r.get("evidence_level") != "Low relevance"]
        if not useful:
            rows.append({
                **meta,
                "pmid": "",
                "title": "",
                "abstract": "",
                "journal": "",
                "year": "",
                "authors": "",
                "publication_types": "",
                "pubmed_url": "",
                "query_type": "",
                "query": "",
                "evidence_score": 0,
                "evidence_level": "No direct evidence found",
                "direct_drug_evidence": False,
                "background_biology_only": False,
                "score_reasons": "PubMed returned papers, but none passed relevance scoring",
            })
            continue

        for rec in useful:
            rows.append({**meta, **rec})

    QUERY_LOG_FILE.write_text(json.dumps(query_log, indent=2, ensure_ascii=False), encoding="utf-8")
    return pd.DataFrame(rows)


# For quick testing, set max_claims=3. For full run, set max_claims=None.
evidence_df = run_literature_agent(claims_df, max_claims=None)
evidence_df.to_csv(EVIDENCE_TABLE_FILE, index=False)
print(evidence_df.shape)
evidence_df.head()

  0%|          | 0/7 [00:00<?, ?it/s]

(38, 26)


,source_file,drug,claim_type,claim_id,claim,hypothesis,why_interesting,suggested_validation,priority,genes,...,authors,publication_types,pubmed_url,evidence_score,evidence_level,direct_drug_evidence,background_biology_only,score_reasons,query_type,query
0,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,executive_summary,6-Mercaptopurine__executive_summary,Proteomic profiling under 6-mercaptopurine sho...,,,,,CDC7;EXO1;E2F6;HYPOXIA;GDF15;IGFBP1;IL4R;TNFRS...,...,Pitchumoni CS; Rubin A; Das K,Journal Article; Review,https://pubmed.ncbi.nlm.nih.gov/20087199/,70,Strong,True,False,query=drug_mechanism; drug_match; mechanism_ma...,drug_mechanism,((6-Mercaptopurine OR mercaptopurine OR 6-MP O...
1,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,executive_summary,6-Mercaptopurine__executive_summary,Proteomic profiling under 6-mercaptopurine sho...,,,,,CDC7;EXO1;E2F6;HYPOXIA;GDF15;IGFBP1;IL4R;TNFRS...,...,Yan Y; Wang Z; Zhou YL; Gao Z; Ning L; Zhao Y,"Journal Article; Research Support, Non-U.S. Gov't",https://pubmed.ncbi.nlm.nih.gov/37586320/,65,Moderate,True,False,query=drug_mechanism; drug_match; mechanism_ma...,drug_mechanism,((6-Mercaptopurine OR mercaptopurine OR 6-MP O...
2,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,executive_summary,6-Mercaptopurine__executive_summary,Proteomic profiling under 6-mercaptopurine sho...,,,,,CDC7;EXO1;E2F6;HYPOXIA;GDF15;IGFBP1;IL4R;TNFRS...,...,Hasskamp J; Meinhardt C; Patton PH; Timmer A,Journal Article; Systematic Review; Meta-Analysis,https://pubmed.ncbi.nlm.nih.gov/40013523/,62,Moderate,True,False,query=drug_mechanism; drug_match; mechanism_ma...,drug_mechanism,((6-Mercaptopurine OR mercaptopurine OR 6-MP O...
3,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,executive_summary,6-Mercaptopurine__executive_summary,Proteomic profiling under 6-mercaptopurine sho...,,,,,CDC7;EXO1;E2F6;HYPOXIA;GDF15;IGFBP1;IL4R;TNFRS...,...,Prefontaine E; Macdonald JK; Sutherland LR,Journal Article; Meta-Analysis; Systematic Review,https://pubmed.ncbi.nlm.nih.gov/19821270/,62,Moderate,True,False,query=drug_mechanism; drug_match; mechanism_ma...,drug_mechanism,((6-Mercaptopurine OR mercaptopurine OR 6-MP O...
4,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,executive_summary,6-Mercaptopurine__executive_summary,Proteomic profiling under 6-mercaptopurine sho...,,,,,CDC7;EXO1;E2F6;HYPOXIA;GDF15;IGFBP1;IL4R;TNFRS...,...,Sandborn WJ; Feagan BG; Marano C; Zhang H; Str...,"Clinical Trial, Phase II; Clinical Trial, Phas...",https://pubmed.ncbi.nlm.nih.gov/23735746/,44,Background,False,True,query=drug_mechanism; mechanism_match:inflamma...,drug_mechanism,((6-Mercaptopurine OR mercaptopurine OR 6-MP O...


## 9. Summarize evidence by claim

In [19]:
def summarize_one_claim(group: pd.DataFrame) -> pd.Series:
    group = group.sort_values("evidence_score", ascending=False)
    top = group.iloc[0]
    n_papers = int(group["pmid"].astype(str).str.len().gt(0).sum())
    n_strong = int((group["evidence_level"] == "Strong").sum())
    n_moderate = int((group["evidence_level"] == "Moderate").sum())
    n_weak = int((group["evidence_level"] == "Weak").sum())
    n_background = int((group["evidence_level"] == "Background").sum())

    if n_strong:
        overall = "Strong direct evidence"
    elif n_moderate:
        overall = "Moderate direct evidence"
    elif n_weak:
        overall = "Weak direct evidence"
    elif n_background:
        overall = "Background biology only; no direct drug-specific evidence"
    else:
        overall = "No direct evidence found"

    top_papers = []
    for _, r in group.head(5).iterrows():
        if clean_text(r.get("pmid", "")):
            top_papers.append(f"PMID {r.get('pmid')}: {r.get('title')} ({r.get('journal')}, {r.get('year')}) [{r.get('evidence_level')}]")

    return pd.Series({
        "source_file": top.get("source_file", ""),
        "drug": top.get("drug", ""),
        "claim_type": top.get("claim_type", ""),
        "claim_id": top.get("claim_id", ""),
        "claim": top.get("claim", ""),
        "priority": top.get("priority", ""),
        "genes": top.get("genes", ""),
        "mechanism_terms": top.get("mechanism_terms", ""),
        "overall_support": overall,
        "n_papers": n_papers,
        "n_strong": n_strong,
        "n_moderate": n_moderate,
        "n_weak": n_weak,
        "n_background": n_background,
        "best_evidence_score": float(top.get("evidence_score", 0)),
        "top_papers": "\n".join(top_papers),
    })


claim_summary_df = (
    evidence_df.groupby("claim_id", dropna=False)
    .apply(summarize_one_claim)
    .reset_index(drop=True)
    .sort_values(["drug", "claim_type", "best_evidence_score"], ascending=[True, True, False])
    .reset_index(drop=True)
)
claim_summary_df.to_csv(CLAIM_SUMMARY_FILE, index=False)
claim_summary_df.head(20)

,source_file,drug,claim_type,claim_id,claim,priority,genes,mechanism_terms,overall_support,n_papers,n_strong,n_moderate,n_weak,n_background,best_evidence_score,top_papers
0,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,executive_summary,6-Mercaptopurine__executive_summary,Proteomic profiling under 6-mercaptopurine sho...,,CDC7;EXO1;E2F6;HYPOXIA;GDF15;IGFBP1;IL4R;TNFRS...,DNA replication;mitotic;mitosis;E2F;G2M;NF.?kB...,Strong direct evidence,7,1,3,0,3,70.0,PMID 20087199: Pancreatitis in inflammatory bo...
1,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,findings_consistent_with_expected_biology,6-Mercaptopurine__Findings consistent with exp...,Activation of hypoxia-like responses (Hallmark...,,HYPOXIA;LOXL2;NRP2;TYRO3;GDF15,hypoxia;HYPOXIA,Strong direct evidence,8,7,0,0,1,92.0,PMID 41683398: Importance and Involvement of I...
2,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,findings_consistent_with_expected_biology,6-Mercaptopurine__Findings consistent with exp...,Induction of inflammatory/NFκB and cellular st...,,APOPTOSIS;GDF15;IGFBP1;IL4R;TNFRSF12A;JUNB;AGR...,NF.?kB;TNFA;inflammatory;apoptosis;UPR;IGF;INF...,Strong direct evidence,5,1,2,0,2,70.0,PMID 20087199: Pancreatitis in inflammatory bo...
3,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,findings_consistent_with_expected_biology,6-Mercaptopurine__Findings consistent with exp...,Strong suppression of DNA replication and mito...,,CDC7;EXO1;E2F6;E2F8;STIL,cell-cycle;DNA replication;mitotic;mitosis;E2F...,Moderate direct evidence,2,0,2,0,0,65.0,PMID 18628594: Effect of 6-mercaptopurine on r...
4,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,unexpected_but_plausible_findings,6-Mercaptopurine__Unexpected but plausible fin...,Pronounced EMT/ECM remodeling and angiogenesis...,,LOXL2;PLAUR;CEMIP;LAMC3;CTHRC1;NRP2,EMT;ECM remodeling;angiogenesis;EMT/tissue,Strong direct evidence,7,4,1,0,2,79.0,PMID 32210620: AOC1 Contributes to Tumor Progr...
5,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,unexpected_but_plausible_findings,6-Mercaptopurine__Unexpected but plausible fin...,Evidence for adaptive signaling and nutrient u...,,IGFBP1;IGFBP3,IGF;PI3K.?AKT;amino acid import,Moderate direct evidence,7,0,2,0,5,65.0,PMID 29760546: Thiopurines are negatively asso...
6,D:\project8 Robotics and AI enable automation...,6-Mercaptopurine,unexpected_but_plausible_findings,6-Mercaptopurine__Unexpected but plausible fin...,Potential metabolic/transport reprogramming: H...,,HMGCR;SLC22A3;SLC13A5;SLC35C1,sterol;solute;transporter,Moderate direct evidence,2,0,1,0,1,62.0,PMID 36351640: [Pharmacogenomics in hematologi...


## 10. Generate Markdown report

In [20]:
def generate_markdown_report(claim_summary_df: pd.DataFrame, evidence_df: pd.DataFrame, output_file: Path, top_papers_per_claim: int = 5):
    lines = []
    lines.append("# Proteomics Summary Literature Evidence Report\n")
    lines.append(f"Total claims: {len(claim_summary_df)}\n")
    lines.append("\n")

    for drug, drug_df in claim_summary_df.groupby("drug", sort=False):
        lines.append(f"## {drug}\n")
        for _, row in drug_df.iterrows():
            lines.append(f"### {row['claim_type']} | {row['claim_id']}\n")
            lines.append(f"- **Claim:** {row['claim']}\n")
            lines.append(f"- **Overall support:** {row['overall_support']}\n")
            lines.append(f"- **Parsed genes/proteins:** {row.get('genes', '')}\n")
            lines.append(f"- **Parsed mechanism terms:** {row.get('mechanism_terms', '')}\n")
            lines.append(f"- **Evidence counts:** strong={row['n_strong']}, moderate={row['n_moderate']}, weak={row['n_weak']}, background={row['n_background']}, total={row['n_papers']}\n")

            papers = evidence_df[evidence_df["claim_id"].astype(str).eq(str(row["claim_id"]))].sort_values("evidence_score", ascending=False)
            if papers["pmid"].astype(str).str.len().gt(0).any():
                lines.append("\nTop papers:\n")
                for _, p in papers.head(top_papers_per_claim).iterrows():
                    if not clean_text(p.get("pmid", "")):
                        continue
                    abs_snip = clean_text(p.get("abstract", ""))[:400]
                    lines.append(
                        f"- **PMID {p.get('pmid')}** | {p.get('evidence_level')} | score={p.get('evidence_score')} | {p.get('title')} "
                        f"({p.get('journal')}, {p.get('year')})\n"
                        f"  - URL: {p.get('pubmed_url')}\n"
                        f"  - Reasons: {p.get('score_reasons')}\n"
                        f"  - Abstract snippet: {abs_snip}\n"
                    )
            else:
                lines.append("\nTop papers: No direct evidence found from the configured PubMed queries.\n")
            lines.append("\n")

    output_file.write_text("\n".join(lines), encoding="utf-8")


generate_markdown_report(claim_summary_df, evidence_df, MARKDOWN_REPORT_FILE)
print(f"Saved claims: {CLAIMS_TABLE_FILE}")
print(f"Saved evidence table: {EVIDENCE_TABLE_FILE}")
print(f"Saved claim summary: {CLAIM_SUMMARY_FILE}")
print(f"Saved report: {MARKDOWN_REPORT_FILE}")
print(f"Saved query log: {QUERY_LOG_FILE}")

Saved claims: D:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards_MAS_results\test\parsed_summary_claims.csv
Saved evidence table: D:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards_MAS_results\test\literature_evidence_long.csv
Saved claim summary: D:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards_MAS_results\test\literature_evidence_by_claim.csv
Saved report: D:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards_MAS_results\test\literature_evidence_report.md
Saved query log: D:\project8  Robotics and AI enable automation in modern proteomics\drug_id_cards_MAS_results\test\query_log.json
